In [5]:
# import
import sqlite3
import pandas as pd
import time
from datetime import datetime

## connection to our oltp database
conn=sqlite3.connect('unisole_rides.db')
cursor=conn.cursor()

print("OLTP database initialized")

OLTP database initialized




## Full Code

```python
# import
import sqlite3
import pandas as pd
import time
from datetime import datetime

# connection to our oltp database
conn = sqlite3.connect('unisole_rides.db')
cursor = conn.cursor()

print("OLTP database initialized")
```

---

# Step-by-Step Explanation 

---

## 1. `# import`

This line is just a comment.

### What is a comment?

* Anything written after `#` is ignored by Python.
* It is used to explain code to humans.

### Data Engineering View:

In real projects, comments are very important because:

* Pipelines are long and complex.
* Multiple engineers work together.
* You need documentation inside the code.

Example:

```python
# This script loads data into OLTP database
```

---

## 2. `import sqlite3`

### What is this doing?

You are telling Python:

"I want to use SQLite database functionality."

---

### What is SQLite?

SQLite is:

* A lightweight database.
* Stored as a file (`.db`).
* No server required, unlike MySQL or PostgreSQL.

---

### Simple Analogy:

Think of SQLite as:
A smart Excel file that supports SQL queries.

---

### Data Engineering Perspective:

This is an OLTP database system.

OLTP stands for Online Transaction Processing.

Used for:

* Real-time operations.
* Fast inserts and updates.
* Examples include:
  * Booking rides.
  * Payments.
  * Orders.

Your file:

```python
'unisole_rides.db'
```

This is your production-like transactional database.

---

## 3. `import pandas as pd`

### What is happening?

You are importing pandas and renaming it as `pd`.

---

### Why rename?

Instead of writing:

```python
pandas.read_csv()
```

You write:

```python
pd.read_csv()
```

This is faster and is standard practice.

---

### What is Pandas?

Pandas is a library used for:

* Data analysis.
* Data cleaning.
* Working with tables, like Excel.

---

### Data Engineering Perspective:

Pandas is used in:

* ETL pipelines.
* Data transformation.
* Feature engineering.

Example:

```python
df = pd.read_sql("SELECT * FROM rides", conn)
```

Here, you can:

* Pull data from the database.
* Clean it.
* Send it to ML models or a data warehouse.

---

## 4. `import time`

### What is this?

This is a built-in Python module for time-related operations.

---

### Why do we use it?

* To add delays.
* To measure execution time.
* To simulate real-time systems.

---

### Data Engineering Use Case:

* Track pipeline performance.
* Add waits between API calls.
* Simulate stream processing.

Example:

```python
time.sleep(2)
```

This waits for 2 seconds.

---

## 5. `from datetime import datetime`

### What is happening?

You are importing a specific class: `datetime` from the `datetime` module.

---

### Why?

To work with:

* Dates.
* Timestamps.

---

### Real Example:

```python
now = datetime.now()
```

Output:

```
2026-04-28 22:10:45
```

---

### Data Engineering Perspective:

This is critical because:

* Every data record needs a timestamp.
* It is used for:
  * Logs.
  * Event tracking.
  * Time-series data.
  * Data partitioning.

Example in pipelines:

```python
created_at = datetime.now()
```

---

## 6. `## connection to our oltp database`

This is just a comment.

It indicates:

Now we are connecting to the OLTP database.

---

## 7. `conn = sqlite3.connect('unisole_rides.db')`

### What is happening?

You are creating a connection to the database.

---

### Think like this:

You want to talk to the database, and `connect()` is like dialing the number.

---

### What is `'unisole_rides.db'`?

* A file.
* It stores all data, including tables and rows.

---

### Behind the scenes:

If the file doesn't exist, SQLite will create it automatically.

---

### Data Engineering Perspective:

This is the entry point of the data pipeline.

You are connecting to the data source and preparing to read or write data.

---

## 8. `cursor = conn.cursor()`

### What is a cursor?

A cursor is a control tool to execute SQL queries.

---

### Analogy:

* `conn` is the connection (like a phone line).
* `cursor` is the person who talks on the call.

---

### What does the cursor do?

* It runs SQL commands.
* It fetches results.

---

### Example:

```python
cursor.execute("SELECT * FROM rides")
data = cursor.fetchall()
```

---

### Data Engineering Perspective:

The cursor is used for:

* Data ingestion.
* Query execution.
* Writing transactional data.

---

## 9. `print("OLTP database initialized")`

### What is happening?

This line prints a message to the console.

---

### Why is this important?

In real systems, it helps with debugging and confirms the system is working.

---

### Output:

```
OLTP database initialized
```

---

### Data Engineering Perspective:

In pipelines, logs like this are important. Instead of using print, we use logging systems such as:

* Airflow logs.
* CloudWatch.
* Datadog.

---

# Big Picture

This small piece of code is actually doing a big thing.

---

## What you just built:

### Step 1: Imported tools

* Database.
* Data processing.
* Time handling.

---

### Step 2: Connected to the OLTP system

```python
unisole_rides.db
```

This simulates a ride booking system, like Uber or Ola.

---

### Step 3: Prepared the query execution layer

```python
cursor
```

---

### Step 4: Initialized the system

```python
print()
```

---

# Real Industry Mapping

| Code       | Real System               |
| ---------- | ------------------------- |
| sqlite3    | MySQL / PostgreSQL        |
| .db file   | Production database       |
| cursor     | Query engine              |
| pandas     | Data transformation layer |
| datetime   | Event tracking            |
| connection | Data source integration   |

---


In [6]:
datetime.now()

datetime.datetime(2026, 4, 28, 21, 58, 24, 817453)

In [7]:
## creating tables with constarints
cursor.execute(''' 
CREATE TABLE IF NOT EXISTS rides(
   ride_id INTEGER PRIMARY KEY AUTOINCREMENT,
   user_id TEXT NOT NULL,
    driver_id TEXT,
    pickup_location TEXT,
    destination TEXT,
    status TEXT CHECK( status IN 
    ('requested','ongoing','compeletd','cancelled')),
    price REAL,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')
conn.commit()
print("table created")

table created


In [10]:
def book_ride(user_id,pickup,dest,price):
    try:
        #start Transaction
        cursor.execute("BEGIN TRANSACTION;")
         # STEP 1 INSERT RIDE 
        cursor.execute(''' 
             INSERT INTO rides(
                       user_id,
                       pickup_location,
                       destination,
                       status,
                       price
        )
        VALUES(?,?,?,'requested',?)
        ''',(user_id,pickup,dest,price))

       ## simulte payment
        payment_success = True
        if  not payment_success:
            raise Exception("payment failed")
        # save changes 
        conn.commit()
        print(f"ride book successfully for {user_id}")

    except Exception as e:
        # rollaback if anythings fails
        conn.rollback()
        print(f"transaction failed due to {e} . No data was saved ")

In [13]:
book_ride("nikita_102","mandi","hamirpur",270)

ride book successfully for nikita_102


In [14]:
def assign_driver(ride_id,driver_id):
    cursor.execute(
        "SELECT status FROM rides WHERE ride_id=?",(ride_id,)
    )
    status= cursor.fetchone()[0]

    if status == 'requested':
        cursor.execute(
            '''
            UPDATE rides 
            SET driver_id=?, status='ongoing'
            WHERE ride_id =?
        ''',
        (driver_id,ride_id)
        )

        conn.commit()
        print(f"Driver {driver_id} assigned to ride {ride_id}")
    else :
        print("ride already assigned or cancelled")
    


In [15]:
assign_driver(1,"driver_deepak")

Driver driver_deepak assigned to ride 1


In [16]:
def get_features_model(user_id):
    query = f"""
    SELECT AVG(price)
    FROM rides
    WHERE user_id = '{user_id}'
    AND status = 'completed'
    LIMIT 5
  """
    df = pd.read_sql_query(query,conn)
    return df.iloc[0,0]